# brz_ytdlp · Google Colab Spike

**Goal**: validate whether Google Colab's free GCP IPs are throttled by YouTube less aggressively than home IPs.

**Hypothesis**: Colab runs on Google infrastructure → YouTube might give Google-egress IPs higher rate limits (or might give them harder limits — we need to measure).

**Test plan** (~10 minutes runtime):
1. Install deps (yt-dlp, langdetect)
2. Probe network: IP, geolocation, youtube.com latency
3. Pull a few BR seeds from a known list (no local DB dependency)
4. Sanity test: validate 5 channels, measure latency + success rate
5. Throughput test: 50 channels × 10 workers concurrent — measure sustained val/s
6. BFS discovery test: run watchEndpoint on 5 huge BR seeds, measure new cid yield
7. Report: comparison vs reference home benchmarks

**Reference home benchmarks** (from local Mac scraper, 2026-05):
- val rate: 5-8 ch/s (single hop, healthy IP)
- watchnext yield: 5.13 new/seed (sustained)
- youtube.com latency: 1-3s through Clash

**Decision criteria**:
- If Colab ≥ home → spike up to multi-Colab parallel scraping farm
- If Colab ≈ home → useful as supplementary, not primary
- If Colab << home (throttled) → abandon Colab path

## 1. Setup — install deps

In [ ]:
!pip install -q yt-dlp langdetect langid

## 2. Network probe — IP / geolocation / latency

In [ ]:
import urllib.request, json, time, socket

def probe():
    # 1. Public IP
    ip = urllib.request.urlopen('https://ifconfig.me', timeout=10).read().decode().strip()
    print(f'  Public IP: {ip}')
    # 2. Geolocation via ipinfo (free, no key for casual queries)
    try:
        info = json.loads(urllib.request.urlopen(f'https://ipinfo.io/{ip}/json', timeout=10).read())
        print(f'  org:      {info.get("org", "?")}')
        print(f'  city:     {info.get("city", "?")}, {info.get("region", "?")}, {info.get("country", "?")}')
    except Exception as e:
        print(f'  geo lookup failed: {e}')
    # 3. youtube.com latency (TCP+TLS+HTTP)
    print(f'\n  youtube.com latency (3 trials):')
    for i in range(3):
        t0 = time.time()
        resp = urllib.request.urlopen('https://www.youtube.com/', timeout=15)
        sz = len(resp.read())
        print(f'    trial {i+1}: {(time.time()-t0)*1000:.0f}ms  ({sz:,} bytes)')
    # 4. InnerTube API latency (closer to what our scraper actually does)
    print(f'\n  InnerTube API (POST /youtubei/v1/browse):')
    body = json.dumps({
        'context': {'client': {'clientName': 'TVHTML5', 'clientVersion': '7.20260114.12.00', 'hl': 'pt-BR', 'gl': 'BR'}},
        'browseId': 'UCr4ARxgElIO21GWfIraZezg'  # Garena FF Brasil (large BR channel)
    }).encode()
    req = urllib.request.Request(
        'https://www.youtube.com/youtubei/v1/browse?prettyPrint=false',
        data=body, headers={'Content-Type': 'application/json', 'User-Agent': 'Mozilla/5.0 (ChromiumStylePlatform) Cobalt/25.lts.30.1034943'}
    )
    for i in range(3):
        t0 = time.time()
        resp = urllib.request.urlopen(req, timeout=15)
        sz = len(resp.read())
        print(f'    trial {i+1}: {(time.time()-t0)*1000:.0f}ms  ({sz:,} bytes)')

probe()

## 3. Pull BR seeds (no DB needed — hardcoded list of known BR channels)

In [ ]:
# 20 known huge BR channels for testing (subs >= 1M, confirmed Brazilian).
# These are PUBLIC channel IDs — no privacy concern. Hardcoded to avoid DB sync.
BR_SEEDS = [
    'UCr4ARxgElIO21GWfIraZezg',  # Garena FF Brasil
    'UCFCUSOunpAQFpFl-M94sX_Q',  # MONTORO BR
    'UCJ0-OtVpF0wOKEqT2Z1HEtA',  # ElectroBOOM (Canada — sanity check)
    'UCTl3QQTvqHFjurroKxexy2Q',  # Olympics (lang_recovered case)
    'UCXBJSI3pCQVnVytwrICafZQ',  # Felipe Neto
    'UCcbexgnP4xCgaqOJsP3uvog',  # Whindersson Nunes
    'UCT9zcQNlyht7fRlcjmflRSA',  # Authentic Games
    'UC8I2Ly20vH_zX0WfHwoY-aQ',  # Castro Brothers
    'UCOFNbqsfRfxuOIRkUbiHmig',  # Cocielo
    'UCo71UNvg3GhPnpD0CkUv24A',  # 5incoMinutos
]
print(f'Loaded {len(BR_SEEDS)} test seeds')

## 4. Sanity test — validate 5 channels, measure per-call latency

In [ ]:
import yt_dlp, time, random, string

def gen_visitor_data():
    """Random visitor_data to avoid session correlation."""
    import base64
    blob = ''.join(random.choices(string.ascii_letters + string.digits + '_-', k=22)).encode()
    return base64.urlsafe_b64encode(blob).decode().rstrip('=')

CLIENTS = ['tv', 'web_safari', 'ios', 'android_vr', 'mweb']

def make_ydl():
    return yt_dlp.YoutubeDL({
        'quiet': True, 'no_warnings': True, 'skip_download': True,
        'socket_timeout': 30,
        'geo_bypass': True, 'geo_bypass_country': 'BR',
        'extractor_args': {
            'youtube': {
                'player_client': [random.choice(CLIENTS)],
                'player_skip': ['js', 'configs'],
                'visitor_data': [gen_visitor_data()],
            },
            'youtubetab': {'skip': ['webpage']},
        },
        'http_headers': {'Accept-Language': 'pt-BR,pt;q=0.9'},
    })

print('Sanity test: hit Stage 1 InnerTube browse for 5 channels')
print(f'{"channel":28s} {"client":12s} {"ms":>5s} {"size":>9s} {"err":40s}')
print('-' * 100)
for cid in BR_SEEDS[:5]:
    ydl = make_ydl()
    client = ydl.params['extractor_args']['youtube']['player_client'][0]
    t0 = time.time()
    try:
        ie = ydl.get_info_extractor('YoutubeTab')
        resp = ie._call_api(ep='browse', video_id=cid, query={'browseId': cid})
        ms = int((time.time()-t0)*1000)
        sz = len(str(resp))
        print(f'{cid[:28]:28s} {client:12s} {ms:>5d} {sz:>9,d}')
    except Exception as e:
        ms = int((time.time()-t0)*1000)
        print(f'{cid[:28]:28s} {client:12s} {ms:>5d} {"-":>9s} {type(e).__name__}: {str(e)[:40]}')
    finally:
        try: ydl.close()
        except: pass

## 5. Throughput test — 50 channels concurrent (10 workers)

This mimics our home production_v2 load pattern at smaller scale. Goal: measure ch/s sustained.

We'll discover ~50 channels via search first (so they're fresh, not just our seeds), then validate them concurrently.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# Step 1: discover ~50 BR channels via search (mirrors home discovery)
BR_QUERIES = [
    'vlog familia brasil', 'receita fácil', 'tutorial android',
    'futebol brasileiro', 'humor stand up', 'sertanejo cover',
    'evangelho oração', 'culinária mineira',
]

def discover_channels(query, limit=30):
    ydl = make_ydl()
    try:
        ie = ydl.get_info_extractor('YoutubeTab')
        resp = ie._call_api(ep='search', video_id='seed', query={'query': query, 'params': 'EgIQAg%3D%3D'})
        cids = []
        # Walk for channelRenderer (channel filter result)
        def walk(n):
            if isinstance(n, dict):
                if 'channelRenderer' in n:
                    cid = n['channelRenderer'].get('channelId')
                    if cid and cid.startswith('UC') and cid not in cids:
                        cids.append(cid)
                for v in n.values(): walk(v)
            elif isinstance(n, list):
                for x in n: walk(x)
        walk(resp)
        return cids[:limit]
    except Exception as e:
        return []
    finally:
        try: ydl.close()
        except: pass

print('Discovering 50 channels via 8 BR search queries...')
all_cids = set()
t0 = time.time()
for q in BR_QUERIES:
    cids = discover_channels(q)
    all_cids.update(cids)
all_cids = list(all_cids)[:50]
print(f'  discovered {len(all_cids)} unique cids in {time.time()-t0:.1f}s')

# Step 2: validate them concurrently — measure throughput
def validate(cid):
    ydl = make_ydl()
    t0 = time.time()
    try:
        ie = ydl.get_info_extractor('YoutubeTab')
        resp = ie._call_api(ep='browse', video_id=cid, query={'browseId': cid})
        return ('ok', cid, int((time.time()-t0)*1000), len(str(resp)))
    except Exception as e:
        return ('err', cid, int((time.time()-t0)*1000), f'{type(e).__name__}: {str(e)[:40]}')
    finally:
        try: ydl.close()
        except: pass

print(f'\nValidating {len(all_cids)} channels with 10 concurrent workers...')
t0 = time.time()
ok_count = 0
err_count = 0
latencies = []
with ThreadPoolExecutor(max_workers=10) as ex:
    futs = [ex.submit(validate, c) for c in all_cids]
    for f in as_completed(futs, timeout=300):
        kind, cid, ms, info = f.result()
        if kind == 'ok':
            ok_count += 1
            latencies.append(ms)
        else:
            err_count += 1
elapsed = time.time() - t0
rate = len(all_cids) / max(0.001, elapsed)

print(f'\n=== Throughput results ===')
print(f'  total: {len(all_cids)} channels in {elapsed:.1f}s')
print(f'  rate:  {rate:.2f} ch/s')
print(f'  ok:    {ok_count}  ({ok_count/len(all_cids)*100:.0f}%)')
print(f'  err:   {err_count} ({err_count/len(all_cids)*100:.0f}%)')
if latencies:
    latencies.sort()
    print(f'  per-call latency: p50={latencies[len(latencies)//2]}ms  p95={latencies[int(len(latencies)*0.95)]}ms')

print(f'\n=== vs home reference ===')
HOME_BENCH = 5.0  # ch/s from local mac with single-hop Clash
delta = rate / HOME_BENCH
if delta >= 1.5:
    print(f'  ✅ Colab {rate:.2f} ch/s vs home {HOME_BENCH:.1f} ch/s = {delta:.2f}x — BETTER, worth scaling!')
elif delta >= 0.7:
    print(f'  🟡 Colab {rate:.2f} ch/s vs home {HOME_BENCH:.1f} ch/s = {delta:.2f}x — similar, supplementary value')
else:
    print(f'  ❌ Colab {rate:.2f} ch/s vs home {HOME_BENCH:.1f} ch/s = {delta:.2f}x — slower, Colab path not worth it')

## 6. BFS test — watchEndpoint discovery on 5 huge BR seeds

Measures whether Colab can productively do the same BFS discovery as our home scraper.

In [ ]:
def discover_watchnext(seed_cid, n_videos=3):
    """Replica of discover_via_watchnext from local scraper (simplified)."""
    owners = set([seed_cid])
    # Step 1: get top N videos from seed's videos tab
    ydl = make_ydl()
    try:
        ie = ydl.get_info_extractor('YoutubeTab')
        resp = ie._call_api(ep='browse', video_id=seed_cid,
                            query={'browseId': seed_cid, 'params': 'EgZ2aWRlb3M%3D'},
                            default_client='web_safari')
        # find video IDs
        video_ids = []
        def walk_for_vid(n):
            if isinstance(n, dict):
                if 'videoRenderer' in n:
                    vid = n['videoRenderer'].get('videoId')
                    if vid and vid not in video_ids: video_ids.append(vid)
                elif 'richItemRenderer' in n:
                    inner = n['richItemRenderer'].get('content', {}).get('videoRenderer', {})
                    if isinstance(inner, dict):
                        vid = inner.get('videoId')
                        if vid and vid not in video_ids: video_ids.append(vid)
                for v in n.values(): walk_for_vid(v)
            elif isinstance(n, list):
                for x in n: walk_for_vid(x)
        walk_for_vid(resp)
        video_ids = video_ids[:n_videos]
    except Exception as e:
        return owners, f'videos tab err: {e}'
    finally:
        try: ydl.close()
        except: pass

    # Step 2: call /next on each video, harvest owner cids
    for vid in video_ids:
        ydl = make_ydl()
        try:
            ie = ydl.get_info_extractor('YoutubeTab')
            n_resp = ie._call_api(ep='next', video_id=vid, query={'videoId': vid}, default_client='web_safari')
            try:
                sec = n_resp['contents']['twoColumnWatchNextResults']['secondaryResults']['secondaryResults']['results']
            except (KeyError, TypeError):
                continue
            for entry in sec:
                lvm = entry.get('lockupViewModel') if isinstance(entry, dict) else None
                if not isinstance(lvm, dict): continue
                try:
                    cid = lvm['metadata']['lockupMetadataViewModel']['image']['decoratedAvatarViewModel']['rendererContext']['commandContext']['onTap']['innertubeCommand']['browseEndpoint']['browseId']
                    if cid and cid.startswith('UC'):
                        owners.add(cid)
                except (KeyError, TypeError):
                    pass
        except Exception:
            continue
        finally:
            try: ydl.close()
            except: pass
    owners.discard(seed_cid)
    return list(owners), None

print('Running watchEndpoint BFS on 5 huge BR seeds (4 API calls each = 20 total)...')
t0 = time.time()
all_discovered = set()
for seed in BR_SEEDS[:5]:
    cids, err = discover_watchnext(seed, n_videos=3)
    print(f'  {seed}: {len(cids)} cids harvested  (err: {err})')
    all_discovered.update(cids)
elapsed = time.time() - t0

print(f'\n=== BFS results ===')
print(f'  total unique cids discovered: {len(all_discovered)}')
print(f'  avg per seed: {len(all_discovered)/5:.1f}')
print(f'  elapsed: {elapsed:.1f}s ({len(all_discovered)/elapsed:.2f} new cid/s)')
print(f'\n  vs home: 5.13 new/seed sustained (after dedup)')
print(f'  Colab sees fresh DB (no dedup) so yield will be higher')

## 7. Multi-session test — open a 2nd Colab notebook, run Cell 2 separately

**Manual step**:
1. Open a 2nd Colab tab (use the **same Google account** or different — both work)
2. Copy Cell 2 (the `probe()` cell) into the 2nd notebook and run it
3. Compare the 'Public IP' lines:
   - Same IP → Colab routes via shared NAT (no parallelization benefit)
   - Different IP → Each Colab session has its own egress IP! Can parallelize 3-5 notebooks for 3-5x throughput

**Why this matters**: If different IPs, the Colab scraper farm idea is viable. If same IP, we'd hit YouTube's per-IP rate limit even with multiple notebooks.

## 8. Verdict & decision

Compare results above against home reference:

| Metric | Home (Mac+Clash) | Colab (this run) | Verdict |
|---|---|---|---|
| youtube.com latency | 1-3s | (Cell 2) | ? |
| InnerTube /browse | 2-3s | (Cell 2) | ? |
| Throughput (10w) | ~5 ch/s | (Cell 5) | ? |
| Multi-session IPs | n/a | (Cell 7) | ? |

If Colab is competitive or better:
- Spin up 3-5 Colab notebooks in parallel
- Each handles a slice of the work
- Sync results via Google Drive / a webhook to home DB

If Colab is slower:
- Move on to other paths: cookie pool / IPv6 VPS / mobile hotspot rotation